In [1]:
from data_utils import HarmonicGraphDataset, harmonic_graph_collate_fn
import GridMLM_tokenizers
from GridMLM_tokenizers import CSGridMLMTokenizer
import torch
from torch.utils.data import DataLoader

from models_graph import HarmonicGraphEncoder
from generate_utils import load_AttnFiLMSEModel

import pickle

/home/maximos/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load the dataset and the tokenizer
with open("data/gjt_train.pkl", "rb") as f:
    gjt_train = pickle.load(f)

tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

In [3]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cpu")

graph_model = HarmonicGraphEncoder()
graph_model.to(device)

# load the model
model = load_AttnFiLMSEModel(
    tokenizer=tokenizer,
    guidance_dim=graph_model.output_dim
)

In [4]:
dataset = HarmonicGraphDataset(gjt_train, tokenizer, model)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=harmonic_graph_collate_fn)

In [5]:
# d = dataset[246]
# print(d)

In [6]:
batch = next(iter(dataloader)) # TODO: check idx: 246 - bar_start: 15 bar_end: 16
print(batch.keys())

dict_keys(['pianoroll', 'real_harmony_ids', 'recomposed_harmony_ids', 'random_harmony_ids', 'real_graph', 'recomposed_graph', 'random_graph', 'mask_token_positions'])


In [19]:
print(batch['pianoroll'].shape)
guide_z = graph_model(batch['real_graph'].to(model.device))
print(guide_z.shape)
constraints = batch['random_harmony_ids'].clone()
constraints[batch['mask_token_positions']] = tokenizer.mask_token_id
print(constraints.shape)
print(constraints.dtype)

torch.Size([16, 80, 13])
torch.Size([16, 128])
torch.Size([16, 80])
torch.int64


In [20]:
logits = model(
    batch['pianoroll'].to(model.device),
    constraints.to(model.device), # TODO: masked version - DONE: 'mask_token_positions' bool
    guide_z.to(model.device)
)
# TODO: when training with graph guidance, we can train for all graph segment
# masked tokens in parallel